# Complaint Intelligence – Full Backend Source Code

This notebook collects **every backend Python source file** of the
*Complaint-Intelligence* project (the `backend/app` package) in one place,
in a logical reading order (config → core → utils → vector store → RAG →
services → schemas → API routes → app entry point → evaluation →
optimization).

**How it works**
- Each code cell starts with `%%writefile backend/<path>`, so running the
  notebook top-to-bottom re-creates the exact original project folder
  structure on disk (nothing is summarized or shortened — every line of
  the original files is preserved).
- Every code cell is preceded by a short **English explanation** describing
  what that file/module does.

**Project summary**: a Retrieval-Augmented-Generation (RAG) chatbot API
(built with FastAPI) that answers questions about a consumer-complaints
dataset. It embeds complaint text with Voyage AI, stores/retrieves vectors
with Pinecone, and generates answers through a chain of free OpenRouter
LLMs (with automatic fallback if one model fails). It also ships
evaluation (BLEU/ROUGE, Recall@K/MRR) and optimization (prompt/embedding/
chunk/strategy comparison) tooling.


## 0. Package markers (`__init__.py`)

The lines below simply (re)create the empty `__init__.py` files that turn
every backend folder into an importable Python package. They contain no
logic — they only exist so that `import app.xxx.yyy` works.


In [ ]:
import os

# Re-create every (empty) package marker file so the folder
# structure behaves as a proper Python package tree.
for rel_path in [
    "backend/app/__init__.py",
    "backend/app/api/__init__.py",
    "backend/app/config/__init__.py",
    "backend/app/core/__init__.py",
    "backend/app/evaluation/__init__.py",
    "backend/app/optimization/__init__.py",
    "backend/app/rag/__init__.py",
    "backend/app/schemas/__init__.py",
    "backend/app/services/__init__.py",
    "backend/app/utils/__init__.py",
    "backend/app/vector_store/__init__.py",
]:
    os.makedirs(os.path.dirname(rel_path), exist_ok=True)
    open(rel_path, "a").close()

## 1. Configuration

### `app/config/settings.py`
Central application configuration, loaded from environment variables / a `.env` file via Pydantic `BaseSettings`. Defines app metadata, the OpenRouter generation model chain (primary + fallback free models), Voyage AI embedding settings, the CSV data path, Pinecone vector-store settings (index, namespace, cloud/region/metric), retrieval defaults (top-K, search type), and CORS origins. `get_settings()` returns a cached singleton `Settings` instance.

In [ ]:
%%writefile backend/app/config/settings.py
"""
Application configuration.

All values are sourced from environment variables so the exact same codebase
runs unmodified locally, in Docker, or on Replit / any cloud host. Both the
vector store (Pinecone) and the embedding model (Voyage AI) are hosted
services -- nothing is downloaded or run locally, so ingestion and serving
work the same on a resource-constrained container as on a laptop.
"""
from functools import lru_cache
from pathlib import Path
from typing import Optional

from pydantic_settings import BaseSettings, SettingsConfigDict

BACKEND_DIR = Path(__file__).resolve().parents[2]
PROJECT_ROOT = BACKEND_DIR.parent


class Settings(BaseSettings):
    """Central application settings, loaded from environment / .env file."""

    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", extra="ignore")

    # --- App metadata ---
    APP_NAME: str = "Consumer Complaints RAG Chatbot API"
    APP_VERSION: str = "2.1.0"
    ENVIRONMENT: str = "development"  # development | production
    LOG_LEVEL: str = "INFO"

    # --- OpenRouter (sole generation provider -- free-tier model chain) ---
    OPENROUTER_API_KEY: Optional[str] = None
    OPENROUTER_BASE_URL: str = "https://openrouter.ai/api/v1/chat/completions"
    OPENROUTER_PRIMARY_MODEL: str = "meta-llama/llama-3.2-3b-instruct:free"
    OPENROUTER_FALLBACK_MODELS: str = "mistralai/mistral-small-2506,deepseek/deepseek-chat,google/gemma-3-4b-it:free"

    @property
    def openrouter_models_list(self) -> list[str]:
        """Ordered chain tried in turn: primary, then each fallback."""
        chain = [self.OPENROUTER_PRIMARY_MODEL]
        chain += [m.strip() for m in self.OPENROUTER_FALLBACK_MODELS.split(",") if m.strip()]
        return chain

    # --- Embeddings (Voyage AI -- hosted API, no local model download) ---
    VOYAGE_API_KEY: Optional[str] = None
    EMBEDDING_MODEL: str = "voyage-3"
    EMBEDDING_DIMENSION: int = 1024  # voyage-3 output size; must match the Pinecone index
    VOYAGE_BATCH_SIZE: int = 128  # Voyage API accepts up to 128 texts per embed call

    # --- Data path ---
    DATA_CSV_PATH: str = str(BACKEND_DIR / "data" / "processed_corpus_5000.csv")

    # --- Pinecone (vector store) ---
    PINECONE_API_KEY: Optional[str] = None
    PINECONE_INDEX_NAME: str = "complaints-rag"
    PINECONE_NAMESPACE: str = "default"
    PINECONE_CLOUD: str = "aws"
    PINECONE_REGION: str = "us-east-1"
    PINECONE_METRIC: str = "cosine"

    # --- Vector store build parameters ---
    MAX_ROWS: Optional[int] = 5000
    BATCH_SIZE: int = 128  # Pinecone upsert batches, matched to VOYAGE_BATCH_SIZE

    # --- Retrieval defaults ---
    RETRIEVER_TOP_K: int = 3
    RETRIEVER_SEARCH_TYPE: str = "similarity"

    # --- CORS ---
    CORS_ORIGINS: str = "*"

    @property
    def cors_origins_list(self) -> list[str]:
        if self.CORS_ORIGINS == "*":
            return ["*"]
        return [o.strip() for o in self.CORS_ORIGINS.split(",") if o.strip()]


@lru_cache
def get_settings() -> Settings:
    """Cached settings instance (avoids re-parsing env on every call)."""
    return Settings()


## 2. Core (exceptions & logging)

### `app/core/exceptions.py`
Defines the application's custom exception hierarchy (`AppException` base class plus `VectorStoreNotReadyError`, `GenerationError`, `ConfigurationError`) and two FastAPI exception handlers that turn any raised `AppException` (or any unhandled exception) into a clean JSON error response instead of a raw traceback.

In [ ]:
%%writefile backend/app/core/exceptions.py
"""Custom application exceptions and FastAPI exception handlers."""
from fastapi import Request, status
from fastapi.responses import JSONResponse


class AppException(Exception):
    """Base exception for all application-level errors."""

    def __init__(self, message: str, status_code: int = status.HTTP_500_INTERNAL_SERVER_ERROR):
        self.message = message
        self.status_code = status_code
        super().__init__(message)


class VectorStoreNotReadyError(AppException):
    """Raised when the Pinecone index has not been built/populated yet."""

    def __init__(self, message: str = "Vector store is not ready. Build it first via the ingestion step."):
        super().__init__(message, status_code=status.HTTP_503_SERVICE_UNAVAILABLE)


class GenerationError(AppException):
    """Raised when the Gemini generation call fails."""

    def __init__(self, message: str = "Failed to generate a response from the language model."):
        super().__init__(message, status_code=status.HTTP_502_BAD_GATEWAY)


class ConfigurationError(AppException):
    """Raised when required configuration (e.g. API keys) is missing."""

    def __init__(self, message: str = "Missing or invalid configuration."):
        super().__init__(message, status_code=status.HTTP_500_INTERNAL_SERVER_ERROR)


async def app_exception_handler(request: Request, exc: AppException) -> JSONResponse:
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": exc.__class__.__name__, "message": exc.message},
    )


async def unhandled_exception_handler(request: Request, exc: Exception) -> JSONResponse:
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={"error": "InternalServerError", "message": "An unexpected error occurred."},
    )


### `app/core/logging_config.py`
Centralized logging setup. `configure_logging()` configures the root logger with a consistent timestamp/level/name/message format, writes to stdout, and quiets down noisy third-party loggers (httpx, pinecone, sentence_transformers, urllib3). `get_logger(name)` is the helper every other module uses to obtain a named logger.

In [ ]:
%%writefile backend/app/core/logging_config.py
"""Centralized logging configuration."""
import logging
import sys

from app.config.settings import get_settings


def configure_logging() -> None:
    """Configure root logging once, for consistent formatting across the app."""
    settings = get_settings()
    level = getattr(logging, settings.LOG_LEVEL.upper(), logging.INFO)

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(formatter)

    root = logging.getLogger()
    root.setLevel(level)
    root.handlers = [handler]

    for noisy in ("httpx", "pinecone", "sentence_transformers", "urllib3"):
        logging.getLogger(noisy).setLevel(logging.WARNING)


def get_logger(name: str) -> logging.Logger:
    return logging.getLogger(name)


## 3. Utilities (data loading)

### `app/utils/data_loader.py`
Loads the cleaned/processed complaints CSV corpus into a pandas DataFrame and converts rows into LangChain `Document` objects (page_content = the pre-built `rag_document` text, metadata = `complaint_id` + `company`) that are later embedded and stored in Pinecone. Also exposes a helper that returns the processed DataFrame (rather than Documents) for building retrieval ground-truth labels used in evaluation.

In [ ]:
%%writefile backend/app/utils/data_loader.py
"""
Loads the cleaned corpus and turns rows into LangChain ``Document`` objects
for Pinecone ingestion.
"""
from typing import List, Optional

import pandas as pd
from langchain_core.documents import Document

from app.config.settings import get_settings
from app.core.logging_config import get_logger

logger = get_logger(__name__)


def load_processed_corpus(csv_path: Optional[str] = None) -> pd.DataFrame:
    settings = get_settings()
    path = csv_path or settings.DATA_CSV_PATH
    df = pd.read_csv(path)
    logger.info("Loaded processed corpus: %s rows from %s", len(df), path)
    return df


def dataframe_to_documents(df: pd.DataFrame, max_rows: Optional[int] = None) -> List[Document]:
    df_processed = df.dropna(subset=["rag_document"]).copy()
    if max_rows is not None:
        df_processed = df_processed.head(max_rows)

    documents = [
        Document(
            page_content=row["rag_document"],
            metadata={
                "complaint_id": str(row["Complaint ID"]),
                "company": str(row["Company"]),
            },
        )
        for _, row in df_processed.iterrows()
    ]
    return documents


def load_documents(csv_path: Optional[str] = None, max_rows: Optional[int] = None) -> List[Document]:
    settings = get_settings()
    df = load_processed_corpus(csv_path)
    effective_max_rows = settings.MAX_ROWS if max_rows is None else max_rows
    return dataframe_to_documents(df, max_rows=effective_max_rows)


def load_dataframe_for_ground_truth(csv_path: Optional[str] = None, max_rows: Optional[int] = None) -> pd.DataFrame:
    settings = get_settings()
    df = load_processed_corpus(csv_path)
    df_processed = df.dropna(subset=["rag_document"]).copy()
    effective_max_rows = settings.MAX_ROWS if max_rows is None else max_rows
    if effective_max_rows is not None:
        df_processed = df_processed.head(effective_max_rows)
    return df_processed


## 4. Vector store (Pinecone + Voyage AI)

### `app/vector_store/builder.py`
Builds/populates the Pinecone index using Voyage AI's hosted embedding API (`voyage-3`). Everything runs remotely (no local model download), so this works fine even on small hosts. Provides `get_embeddings()` (Voyage embeddings client), `get_pinecone_client()`, `ensure_index_exists()` (creates the serverless Pinecone index if missing), and `build_vector_store()` which embeds & upserts the documents in memory-safe batches, running garbage collection between batches. Runnable standalone as `python -m app.vector_store.builder`.

In [ ]:
%%writefile backend/app/vector_store/builder.py
"""
Vector store builder -- Pinecone + Voyage AI edition.

Builds (or rebuilds) the Pinecone index using Voyage AI's hosted embedding
API (`voyage-3`), upserted in memory-safe batches. Nothing is downloaded or
computed locally: both the embedding model and the vector store are remote
services, so this runs fine on small/resource-constrained hosts (e.g. a free
Replit container).
"""
import gc
from typing import List, Optional

from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_voyageai import VoyageAIEmbeddings
from pinecone import Pinecone, ServerlessSpec

from app.config.settings import get_settings
from app.core.exceptions import ConfigurationError
from app.core.logging_config import get_logger
from app.utils.data_loader import load_documents

logger = get_logger(__name__)


def get_embeddings(model_name: Optional[str] = None) -> VoyageAIEmbeddings:
    """Voyage AI hosted embeddings -- no local model download or inference."""
    settings = get_settings()
    if not settings.VOYAGE_API_KEY:
        raise ConfigurationError("VOYAGE_API_KEY is not set. Add it to your environment or .env file.")
    return VoyageAIEmbeddings(
        voyage_api_key=settings.VOYAGE_API_KEY,
        model=model_name or settings.EMBEDDING_MODEL,
        batch_size=settings.VOYAGE_BATCH_SIZE,
    )


def get_pinecone_client() -> Pinecone:
    settings = get_settings()
    if not settings.PINECONE_API_KEY:
        raise ConfigurationError("PINECONE_API_KEY is not set. Add it to your environment or .env file.")
    return Pinecone(api_key=settings.PINECONE_API_KEY)


def ensure_index_exists(pc: Optional[Pinecone] = None) -> None:
    """Create the Pinecone serverless index if it doesn't exist yet."""
    settings = get_settings()
    pc = pc or get_pinecone_client()
    existing = {idx["name"] for idx in pc.list_indexes()}
    if settings.PINECONE_INDEX_NAME in existing:
        logger.info("Pinecone index '%s' already exists.", settings.PINECONE_INDEX_NAME)
        return

    logger.info(
        "Creating Pinecone index '%s' (dim=%s, metric=%s, cloud=%s, region=%s)",
        settings.PINECONE_INDEX_NAME,
        settings.EMBEDDING_DIMENSION,
        settings.PINECONE_METRIC,
        settings.PINECONE_CLOUD,
        settings.PINECONE_REGION,
    )
    pc.create_index(
        name=settings.PINECONE_INDEX_NAME,
        dimension=settings.EMBEDDING_DIMENSION,
        metric=settings.PINECONE_METRIC,
        spec=ServerlessSpec(cloud=settings.PINECONE_CLOUD, region=settings.PINECONE_REGION),
    )


def build_vector_store(
    documents: Optional[List[Document]] = None,
    namespace: Optional[str] = None,
    batch_size: Optional[int] = None,
) -> PineconeVectorStore:
    """
    Build/populate the Pinecone index in fixed-size batches: each batch is
    embedded via one Voyage API call, then upserted to Pinecone.
    """
    settings = get_settings()
    namespace = namespace or settings.PINECONE_NAMESPACE
    batch_size = batch_size or settings.BATCH_SIZE

    if documents is None:
        documents = load_documents()

    logger.info("Building Pinecone index from %s documents (batch_size=%s, namespace=%s)",
                len(documents), batch_size, namespace)

    pc = get_pinecone_client()
    ensure_index_exists(pc)

    embeddings = get_embeddings()
    index = pc.Index(settings.PINECONE_INDEX_NAME)
    vector_store = PineconeVectorStore(index=index, embedding=embeddings, namespace=namespace)

    total = len(documents)
    for i in range(0, total, batch_size):
        batch_docs = documents[i : i + batch_size]
        ids = [doc.metadata.get("complaint_id", f"doc-{i + j}") for j, doc in enumerate(batch_docs)]
        vector_store.add_documents(batch_docs, ids=ids)
        logger.info("Batch %s/%s upserted (%s vectors)", (i // batch_size) + 1, -(-total // batch_size), len(batch_docs))
        del batch_docs
        gc.collect()

    logger.info("Pinecone index '%s' populated successfully.", settings.PINECONE_INDEX_NAME)
    return vector_store


if __name__ == "__main__":
    # Standalone ingestion entry point:
    #   python -m app.vector_store.builder
    build_vector_store()


### `app/vector_store/store.py`
Connects to an *already populated* Pinecone index for serving (no rebuild here). `get_vector_store()` returns a cached `PineconeVectorStore` instance (raises `VectorStoreNotReadyError` if API keys are missing or the index doesn't exist yet). `get_retriever(k, search_type)` wraps it as a LangChain retriever (defaults: k=3, similarity search). `vector_store_is_ready()` is a lightweight health-check used by the `/health` endpoint.

In [ ]:
%%writefile backend/app/vector_store/store.py
"""
Loads the Pinecone index for serving (no rebuild), and exposes retrievers
with the same defaults as the original pipeline (k=3, similarity search).
Embeddings are computed via the Voyage AI API -- no local model.
"""
from functools import lru_cache
from typing import Optional

from langchain_core.vectorstores import VectorStoreRetriever
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

from app.config.settings import get_settings
from app.core.exceptions import VectorStoreNotReadyError
from app.core.logging_config import get_logger
from app.vector_store.builder import get_embeddings, get_pinecone_client

logger = get_logger(__name__)


@lru_cache
def get_vector_store() -> PineconeVectorStore:
    """Attach to the existing Pinecone index (must already be populated)."""
    settings = get_settings()

    if not settings.PINECONE_API_KEY:
        raise VectorStoreNotReadyError(
            "PINECONE_API_KEY is not set. Configure it in the environment or .env file."
        )
    if not settings.VOYAGE_API_KEY:
        raise VectorStoreNotReadyError(
            "VOYAGE_API_KEY is not set. Configure it in the environment or .env file."
        )

    pc = get_pinecone_client()
    existing = {idx["name"] for idx in pc.list_indexes()}
    if settings.PINECONE_INDEX_NAME not in existing:
        raise VectorStoreNotReadyError(
            f"Pinecone index '{settings.PINECONE_INDEX_NAME}' does not exist. "
            "Run `python -m app.vector_store.builder` to create and populate it first."
        )

    index = pc.Index(settings.PINECONE_INDEX_NAME)
    embeddings = get_embeddings()
    store = PineconeVectorStore(index=index, embedding=embeddings, namespace=settings.PINECONE_NAMESPACE)
    logger.info("Connected to Pinecone index '%s' (namespace=%s)", settings.PINECONE_INDEX_NAME, settings.PINECONE_NAMESPACE)
    return store


def get_retriever(k: Optional[int] = None, search_type: Optional[str] = None) -> VectorStoreRetriever:
    """Return a retriever. Defaults match the original pipeline (k=3, similarity)."""
    settings = get_settings()
    store = get_vector_store()
    k = k or settings.RETRIEVER_TOP_K
    search_type = search_type or settings.RETRIEVER_SEARCH_TYPE
    return store.as_retriever(search_type=search_type, search_kwargs={"k": k})


def vector_store_is_ready() -> bool:
    """Check Pinecone connectivity + that the index exists and has vectors."""
    settings = get_settings()
    if not settings.PINECONE_API_KEY or not settings.VOYAGE_API_KEY:
        return False
    try:
        pc: Pinecone = get_pinecone_client()
        existing = {idx["name"] for idx in pc.list_indexes()}
        if settings.PINECONE_INDEX_NAME not in existing:
            return False
        index = pc.Index(settings.PINECONE_INDEX_NAME)
        stats = index.describe_index_stats()
        namespace_stats = stats.get("namespaces", {}).get(settings.PINECONE_NAMESPACE, {})
        return namespace_stats.get("vector_count", 0) > 0
    except Exception:  # noqa: BLE001 -- health check must never raise
        logger.exception("Pinecone readiness check failed")
        return False


## 5. RAG pipeline (prompting & generation chain)

### `app/rag/prompts.py`
Prompt-construction helpers. `format_docs()` joins retrieved documents into one context string. `build_default_prompt()` builds the production system/user prompt instructing the model to answer naturally without referencing 'the context'. `get_prompt(version="v1"|"v2")` builds two alternative prompt styles (plain baseline vs. role/constraint-based) used by the prompt-engineering comparison experiment.

In [ ]:
%%writefile backend/app/rag/prompts.py
"""Prompt construction for the RAG chain and prompt-engineering experiments."""


def format_docs(docs) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


def build_default_prompt(context: str, question: str) -> str:
    return (
        f"You are an expert customer support AI assistant specializing in analyzing consumer financial complaints.\n"
        f"Answer the user's question naturally and directly.\n"
        f"Use the retrieved information only as background knowledge.\n"
        f"Do NOT mention the context, retrieved documents, or phrases like "
        f"'Based on the provided context', 'According to the context', or "
        f"'The provided information'.\n"
        f"If the answer is not available, simply say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )


def get_prompt(context: str, question: str, version: str = "v1") -> str:
    """version = 'v1' (baseline) or 'v2' (role/constraint-based) -- used by the prompt-engineering experiment."""
    if version == "v1":
        return f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    return (
        f"You are a professional Financial Complaint Analyst. "
        f"Use ONLY the provided context below to answer. "
        f"If the answer isn't in the context, say 'I do not have enough information.' "
        f"Do not invent facts.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


### `app/rag/chain.py`
The core RAG chain: Pinecone retriever → prompt → generation. Generation goes exclusively through OpenRouter's free-tier models — a primary model plus an ordered list of fallback models (`MODEL_CHAIN`), each tried in turn via `generate_completion()` until one succeeds. `generate_answer_from_context()` builds the final prompt and calls the chain, returning an error message string instead of raising if every model fails. `build_rag_chain()` wires the retriever, prompt formatting and generation into one LangChain Runnable; `answer_question()` is a convenience one-shot call.

In [ ]:
%%writefile backend/app/rag/chain.py
"""
RAG chain: Pinecone retriever -> prompt -> generation.

Generation is entirely via OpenRouter's free-tier models (no Gemini, no
paid provider). A fixed model chain is tried in order until one succeeds:

    PRIMARY_MODEL   = "meta-llama/llama-3.2-3b-instruct:free"
    FALLBACK_MODELS = ["mistralai/mistral-small-2506",
                        "deepseek/deepseek-chat",
                        "google/gemma-3-4b-it:free"]
"""
import httpx
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from app.config.settings import get_settings
from app.core.exceptions import ConfigurationError, GenerationError
from app.core.logging_config import get_logger
from app.rag.prompts import build_default_prompt, format_docs
from app.vector_store.store import get_retriever

logger = get_logger(__name__)

PRIMARY_MODEL = "openai/gpt-oss-20b:free"
FALLBACK_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "google/gemma-4-26b-a4b-it:free",
    "nvidia/nemotron-nano-9b-v2:free",
]
MODEL_CHAIN = [PRIMARY_MODEL, *FALLBACK_MODELS]


def _call_openrouter(full_prompt: str, model: str) -> str:
    settings = get_settings()
    if not settings.OPENROUTER_API_KEY:
        raise ConfigurationError("OPENROUTER_API_KEY is not set. Add it to your environment or .env file.")

    response = httpx.post(
        settings.OPENROUTER_BASE_URL,
        headers={
            "Authorization": f"Bearer {settings.OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": full_prompt}],
        },
        timeout=30.0,
    )
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]


def generate_completion(full_prompt: str) -> str:
    """Walks MODEL_CHAIN in order, returning the first successful response."""
    last_error = None
    for model in MODEL_CHAIN:
        try:
            answer = _call_openrouter(full_prompt, model)
            logger.info("Answered via OpenRouter model '%s'.", model)
            return answer
        except Exception as e:  # noqa: BLE001 -- try the next model in the chain
            logger.warning("OpenRouter model '%s' failed (%s); trying next.", model, e)
            last_error = e

    logger.error("All OpenRouter models in the chain failed.")
    raise GenerationError(f"All OpenRouter models failed. Last error: {last_error}")


def generate_answer_from_context(input_dict: dict) -> str:
    """
    Generation entry point used by the RAG chain and the qualitative-eval
    generation itself runs entirely through the OpenRouter chain above.
    """
    context = input_dict["context"]
    question = input_dict["question"]
    full_prompt = build_default_prompt(context, question)
    try:
        return generate_completion(full_prompt)
    except GenerationError as e:
        return f"Error: {e.message}"


def build_rag_chain():
    retriever = get_retriever()
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | RunnableLambda(generate_answer_from_context)
    )
    return rag_chain


def answer_question(question: str) -> str:
    chain = build_rag_chain()
    return chain.invoke(question)


## 6. Pydantic schemas (request/response models)

### `app/schemas/chat.py`
Request/response models for the `/api/v1/chat` endpoint: `ChatRequest` (the user's question), `SourceDocument` (a retrieved source shown in the UI), and `ChatResponse` (answer + question + list of sources).

In [ ]:
%%writefile backend/app/schemas/chat.py
"""Pydantic schemas for the /api/v1/chat endpoint."""
from typing import List

from pydantic import BaseModel, Field


class ChatRequest(BaseModel):
    question: str = Field(..., min_length=1, description="The user's question")


class SourceDocument(BaseModel):
    complaint_id: str
    company: str
    snippet: str


class ChatResponse(BaseModel):
    answer: str
    question: str
    sources: List[SourceDocument] = []


### `app/schemas/evaluation.py`
Response models for the `/api/v1/evaluation/*` endpoints: generation metrics rows, retrieval metrics rows, the qualitative-run report path/count, and a combined summary response.

In [ ]:
%%writefile backend/app/schemas/evaluation.py
"""Pydantic schemas for the /api/v1/evaluation endpoints."""
from typing import Any, Dict, List

from pydantic import BaseModel


class GenerationMetricsResponse(BaseModel):
    rows: List[Dict[str, Any]] = []


class RetrievalMetricsResponse(BaseModel):
    rows: List[Dict[str, Any]] = []


class QualitativeEvalResponse(BaseModel):
    report_path: str
    num_queries: int


class EvaluationSummaryResponse(BaseModel):
    generation_metrics: List[Dict[str, Any]] = []
    retrieval_metrics: List[Dict[str, Any]] = []


### `app/schemas/health.py`
Response model for the `/api/v1/health` endpoint, reporting service status, vector-store readiness, which backends/models are configured (Pinecone, Voyage, OpenRouter model chain), and whether each required API key is present.

In [ ]:
%%writefile backend/app/schemas/health.py
from pydantic import BaseModel


class HealthResponse(BaseModel):
    status: str
    vector_store_ready: bool
    vector_store_backend: str
    pinecone_index: str
    embedding_model: str
    embedding_backend: str
    generation_backend: str
    generation_model_chain: list[str]
    openrouter_api_key_configured: bool
    pinecone_api_key_configured: bool
    voyage_api_key_configured: bool
    version: str


### `app/schemas/retrieval.py`
Request/response models for the `/api/v1/retrieve` endpoint (retrieval-only, no generation): `RetrievalRequest` (query, optional k, optional search_type) and `RetrievalResponse` (a list of `RetrievedDocument`).

In [ ]:
%%writefile backend/app/schemas/retrieval.py
from typing import List, Optional

from pydantic import BaseModel, Field


class RetrievalRequest(BaseModel):
    query: str = Field(..., min_length=1)
    k: Optional[int] = Field(default=None, ge=1, le=20)
    search_type: Optional[str] = Field(default=None, description="'similarity' or 'mmr'")


class RetrievedDocument(BaseModel):
    complaint_id: str
    company: str
    content: str


class RetrievalResponse(BaseModel):
    query: str
    results: List[RetrievedDocument]


## 7. Services (business logic layer between API routes and RAG/vector store)

### `app/services/chat_service.py`
`ask_question()` — the service backing the chat endpoint: retrieves relevant documents, formats them as context, generates an answer through the RAG chain, and returns a `ChatResponse` that includes trimmed source snippets for the UI's 'sources' panel.

In [ ]:
%%writefile backend/app/services/chat_service.py
"""Chat service -- wraps the RAG chain for the /api/v1/chat endpoint."""
from typing import Optional

from app.core.logging_config import get_logger
from app.rag.chain import generate_answer_from_context
from app.rag.prompts import format_docs
from app.schemas.chat import ChatResponse, SourceDocument
from app.vector_store.store import get_retriever

logger = get_logger(__name__)


def ask_question(question: str, k: Optional[int] = None) -> ChatResponse:
    """Retrieve -> format -> generate, surfacing sources for the UI's context viewer."""
    retriever = get_retriever(k=k)
    docs = retriever.invoke(question)
    context = format_docs(docs)
    answer = generate_answer_from_context({"context": context, "question": question})

    sources = [
        SourceDocument(
            complaint_id=d.metadata.get("complaint_id", "unknown"),
            company=d.metadata.get("company", "unknown"),
            snippet=(d.page_content[:300] + "...") if len(d.page_content) > 300 else d.page_content,
        )
        for d in docs
    ]
    return ChatResponse(answer=answer, question=question, sources=sources)


### `app/services/retrieval_service.py`
`retrieve()` — the service backing the retrieval-only endpoint: runs the retriever directly (no LLM call) and returns the raw retrieved documents, useful for debugging/inspecting what the vector store returns for a given query.

In [ ]:
%%writefile backend/app/services/retrieval_service.py
"""Retrieval service -- exposes the Pinecone retriever directly, for debugging/inspection."""
from typing import Optional

from app.schemas.retrieval import RetrievalResponse, RetrievedDocument
from app.vector_store.store import get_retriever


def retrieve(query: str, k: Optional[int] = None, search_type: Optional[str] = None) -> RetrievalResponse:
    retriever = get_retriever(k=k, search_type=search_type)
    docs = retriever.invoke(query)
    results = [
        RetrievedDocument(
            complaint_id=d.metadata.get("complaint_id", "unknown"),
            company=d.metadata.get("company", "unknown"),
            content=d.page_content,
        )
        for d in docs
    ]
    return RetrievalResponse(query=query, results=results)


### `app/services/evaluation_service.py`
Thin orchestration layer shared by the API and the standalone evaluation scripts: `run_generation_metrics()` (BLEU/ROUGE over the fixed eval dataset), `run_retrieval_metrics()` (Recall@K/MRR at k=1,3,5), and `run_qualitative()` (runs the 9 fixed test queries through the live chain and writes a text report).

In [ ]:
%%writefile backend/app/services/evaluation_service.py
"""Evaluation service -- thin orchestration layer shared by the API and standalone scripts."""
from typing import Any, Dict, List

from app.evaluation.generation_metrics import EVAL_DATASET, run_generation_evaluation
from app.evaluation.retrieval_metrics import TEST_QUERIES, run_retrieval_evaluation
from app.evaluation.run_qualitative import run_qualitative_eval
from app.utils.data_loader import load_dataframe_for_ground_truth
from app.vector_store.store import get_retriever


def run_generation_metrics() -> List[Dict[str, Any]]:
    retriever = get_retriever()
    df = run_generation_evaluation(retriever, EVAL_DATASET)
    return df.to_dict(orient="records")


def run_retrieval_metrics(k_values: List[int] = None) -> List[Dict[str, Any]]:
    retriever = get_retriever()
    df_processed = load_dataframe_for_ground_truth()
    df = run_retrieval_evaluation(retriever, df_processed, k_values=k_values or [1, 3, 5])
    return df.to_dict(orient="records")


def run_qualitative(output_path: str = "./evaluation_report.txt") -> Dict[str, Any]:
    path = run_qualitative_eval(output_path=output_path)
    return {"report_path": path, "num_queries": len(TEST_QUERIES)}


## 8. API routes (FastAPI routers)

### `app/api/chat.py`
`POST /api/v1/chat` — accepts a `ChatRequest`, optionally overrides top-K via a query parameter, logs the incoming question, and delegates to `ask_question()`.

In [ ]:
%%writefile backend/app/api/chat.py
from fastapi import APIRouter, Query

from app.core.logging_config import get_logger
from app.schemas.chat import ChatRequest, ChatResponse
from app.services.chat_service import ask_question

logger = get_logger(__name__)

router = APIRouter(prefix="/api/v1", tags=["Chat"])


@router.post("/chat", response_model=ChatResponse, summary="Ask the RAG chatbot a question")
def chat(
    payload: ChatRequest,
    k: int = Query(default=None, ge=1, le=20, description="Override top-K retrieval"),
) -> ChatResponse:
    logger.info("Chat request: %s", payload.question[:80])
    return ask_question(payload.question, k=k)


### `app/api/retrieval.py`
`POST /api/v1/retrieve` — accepts a `RetrievalRequest` (query, k, search_type) and delegates to the retrieval service, returning matched documents without any generation step.

In [ ]:
%%writefile backend/app/api/retrieval.py
from fastapi import APIRouter

from app.schemas.retrieval import RetrievalRequest, RetrievalResponse
from app.services.retrieval_service import retrieve

router = APIRouter(prefix="/api/v1", tags=["Retrieval"])


@router.post("/retrieve", response_model=RetrievalResponse, summary="Run retrieval only (no generation)")
def retrieve_documents(payload: RetrievalRequest) -> RetrievalResponse:
    return retrieve(payload.query, k=payload.k, search_type=payload.search_type)


### `app/api/health.py`
`GET /api/v1/health` — reports overall service status plus the configured vector-store backend (Pinecone), embedding backend (Voyage AI), generation backend/model chain (OpenRouter), and whether each required API key is set.

In [ ]:
%%writefile backend/app/api/health.py
from fastapi import APIRouter

from app.config.settings import get_settings
from app.rag.chain import MODEL_CHAIN
from app.schemas.health import HealthResponse
from app.vector_store.store import vector_store_is_ready

router = APIRouter(prefix="/api/v1", tags=["Health"])


@router.get("/health", response_model=HealthResponse, summary="Service health check")
def health_check() -> HealthResponse:
    settings = get_settings()
    return HealthResponse(
        status="ok",
        vector_store_ready=vector_store_is_ready(),
        vector_store_backend="pinecone",
        pinecone_index=settings.PINECONE_INDEX_NAME,
        embedding_model=settings.EMBEDDING_MODEL,
        embedding_backend="voyageai",
        generation_backend="openrouter",
        generation_model_chain=MODEL_CHAIN,
        openrouter_api_key_configured=bool(settings.OPENROUTER_API_KEY),
        pinecone_api_key_configured=bool(settings.PINECONE_API_KEY),
        voyage_api_key_configured=bool(settings.VOYAGE_API_KEY),
        version=settings.APP_VERSION,
    )


### `app/api/evaluation.py`
Evaluation endpoints: `GET /api/v1/evaluation/generation` (BLEU/ROUGE), `GET /api/v1/evaluation/retrieval` (Recall@K/MRR), `POST /api/v1/evaluation/qualitative` (runs the 9 fixed test queries and saves a report), and `GET /api/v1/evaluation/summary` (combined generation + retrieval metrics).

In [ ]:
%%writefile backend/app/api/evaluation.py
from fastapi import APIRouter

from app.schemas.evaluation import (
    EvaluationSummaryResponse,
    GenerationMetricsResponse,
    QualitativeEvalResponse,
    RetrievalMetricsResponse,
)
from app.services.evaluation_service import (
    run_generation_metrics,
    run_qualitative,
    run_retrieval_metrics,
)

router = APIRouter(prefix="/api/v1/evaluation", tags=["Evaluation"])


@router.get("/generation", response_model=GenerationMetricsResponse, summary="BLEU / ROUGE generation metrics")
def generation_metrics() -> GenerationMetricsResponse:
    return GenerationMetricsResponse(rows=run_generation_metrics())


@router.get("/retrieval", response_model=RetrievalMetricsResponse, summary="Recall@K / MRR retrieval metrics")
def retrieval_metrics() -> RetrievalMetricsResponse:
    return RetrievalMetricsResponse(rows=run_retrieval_metrics())


@router.post("/qualitative", response_model=QualitativeEvalResponse, summary="Run the 9 fixed test queries and save a report")
def qualitative_eval() -> QualitativeEvalResponse:
    result = run_qualitative()
    return QualitativeEvalResponse(**result)


@router.get("/summary", response_model=EvaluationSummaryResponse, summary="Combined generation + retrieval metrics")
def evaluation_summary() -> EvaluationSummaryResponse:
    return EvaluationSummaryResponse(
        generation_metrics=run_generation_metrics(),
        retrieval_metrics=run_retrieval_metrics(),
    )


## 9. Application entry point

### `app/main.py`
FastAPI application factory and entry point. Configures logging on import, defines an async `lifespan` context (startup/shutdown log messages), builds the `FastAPI` app with CORS middleware, registers the custom exception handlers, includes all routers (health, chat, retrieval, evaluation), and — if a built React frontend exists — mounts it as static files and serves `index.html` for any unmatched path (SPA fallback). The module-level `app` object is what Uvicorn/Gunicorn runs in production.

In [ ]:
%%writefile backend/app/main.py
"""FastAPI application entry point."""
from contextlib import asynccontextmanager
from pathlib import Path

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse

from app.api import chat, evaluation, health, retrieval
from app.config.settings import get_settings
from app.core.exceptions import AppException, app_exception_handler, unhandled_exception_handler
from app.core.logging_config import configure_logging, get_logger

FRONTEND_DIST = Path(__file__).resolve().parents[2] / "frontend" / "dist"

configure_logging()
logger = get_logger(__name__)


@asynccontextmanager
async def lifespan(app: FastAPI):
    settings = get_settings()
    logger.info("Starting %s v%s (%s)", settings.APP_NAME, settings.APP_VERSION, settings.ENVIRONMENT)
    yield
    logger.info("Shutting down %s", settings.APP_NAME)


def create_app() -> FastAPI:
    settings = get_settings()
    app = FastAPI(
        title=settings.APP_NAME,
        version=settings.APP_VERSION,
        description=(
            "Production API for the Consumer Complaints RAG Chatbot. "
            "Wraps the RAG pipeline (Pinecone + HuggingFace embeddings + Gemini), "
            "evaluation (BLEU/ROUGE, Recall@K/MRR), and optimization experiments."
        ),
        lifespan=lifespan,
    )

    app.add_middleware(
        CORSMiddleware,
        allow_origins=settings.cors_origins_list,
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )

    app.add_exception_handler(AppException, app_exception_handler)
    app.add_exception_handler(Exception, unhandled_exception_handler)

    app.include_router(health.router)
    app.include_router(chat.router)
    app.include_router(retrieval.router)
    app.include_router(evaluation.router)

    # Serve built React frontend in production
    if FRONTEND_DIST.exists():
        app.mount("/assets", StaticFiles(directory=FRONTEND_DIST / "assets"), name="assets")

        @app.get("/{full_path:path}", include_in_schema=False)
        async def serve_frontend(full_path: str):
            return FileResponse(FRONTEND_DIST / "index.html")

    return app


app = create_app()


## 10. Evaluation (generation & retrieval quality)

### `app/evaluation/generation_metrics.py`
Generation-quality evaluation using ROUGE-1, ROUGE-L and smoothed sentence-BLEU. Contains a small fixed `EVAL_DATASET` of (question, ground_truth) pairs, `calculate_metrics()` to score a single answer, and `run_generation_evaluation()` which retrieves context, builds the v1 prompt, calls the OpenRouter model chain, times the call, and scores every question in the dataset.

In [ ]:
%%writefile backend/app/evaluation/generation_metrics.py
"""Generation-quality evaluation (ROUGE-1 / ROUGE-L / smoothed sentence-BLEU)."""
import time
from typing import Tuple

import pandas as pd
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer

_scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
_smoothing = SmoothingFunction().method1

EVAL_DATASET = [
    {
        "question": "What are the details of the complaint filed against I.C. System, Inc. regarding an attempt to collect a debt not owed due to identity theft hindering the consumer's ability to close a loan?",
        "ground_truth": "I.C. System, Inc. attempted to collect a debt that the consumer did not owe because it resulted from identity theft. This fraudulent debt severely hindered the consumer's ability to secure or close a loan.",
    },
    {
        "question": "If a consumer complains that Wells Fargo is incorrectly reporting a checking/savings account as a credit card on their credit profile, what is the typical company response or resolution outcome for such an issue?",
        "ground_truth": "Wells Fargo typically responds by reviewing the credit profile reporting error, updating the status, and confirming whether the account is correctly categorized as a banking relationship or closing/correcting the record.",
    },
    {
        "question": "Summarize the primary reasons why consumers file complaints regarding 'Attempts to collect debt not owed' within this dataset sample. What is the most common underlying cause?",
        "ground_truth": "The primary reasons include identity theft, medical bills already paid, or misidentified accounts. The most common underlying cause is outdated or incorrect information in collection agency records.",
    },
]


def calculate_metrics(bot_ans: str, ground_truth: str) -> Tuple[float, float, float]:
    rouge_results = _scorer.score(ground_truth, bot_ans)
    r1, rl = rouge_results["rouge1"].fmeasure, rouge_results["rougeL"].fmeasure
    bleu = sentence_bleu(
        [ground_truth.lower().split()],
        bot_ans.lower().split(),
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=_smoothing,
    )
    return round(r1, 3), round(rl, 3), round(bleu, 3)


def run_generation_evaluation(retriever, eval_dataset=None) -> "pd.DataFrame":
    """Generation now runs through the OpenRouter model chain (see app.rag.chain)."""
    from app.rag.chain import generate_completion
    from app.rag.prompts import get_prompt

    eval_dataset = eval_dataset or EVAL_DATASET
    results_list = []
    for idx, item in enumerate(eval_dataset, 1):
        q, gt = item["question"], item["ground_truth"]
        start_time = time.time()
        docs = retriever.invoke(q)
        context = "\n\n".join(doc.page_content for doc in docs)
        full_prompt = get_prompt(context, q, version="v1")
        answer_text = generate_completion(full_prompt)
        latency = round(time.time() - start_time, 2)
        r1, rl, bleu = calculate_metrics(answer_text, gt)
        results_list.append({"Q": idx, "Latency(s)": latency, "ROUGE-1": r1, "ROUGE-L": rl, "BLEU": bleu})
    return pd.DataFrame(results_list)


### `app/evaluation/retrieval_metrics.py`
Retrieval-quality evaluation (Recall@K, MRR) against programmatically derived ground truth. Contains 9 fixed `TEST_QUERIES` and matching `RETRIEVAL_GROUND_TRUTH` filters (company/issue/narrative substring or regex matches). `get_relevant_ids()`/`build_ground_truth_ids()` turn those filters into sets of relevant complaint IDs from the processed DataFrame. `evaluate_retrieval()` computes Recall@K and MRR for one retriever/k; `run_retrieval_evaluation()` runs this across several k values (default 1, 3, 5).

In [ ]:
%%writefile backend/app/evaluation/retrieval_metrics.py
"""
Retrieval-quality evaluation (Recall@K, MRR against programmatically-derived
ground truth). Works against the Pinecone-backed retriever unchanged, since
`PineconeVectorStore.similarity_search` implements the same interface used
here.
"""
from typing import Dict, List, Set

import pandas as pd

TEST_QUERIES: List[str] = [
    "I have an issue with U.S. BANCORP regarding my mother's closed credit card account which had a credit balance of $420.00, and I have power of attorney. What happened in this specific complaint?",
    "What are the details of the complaint filed against I.C. System, Inc. regarding an attempt to collect a debt not owed due to identity theft hindering the consumer's ability to close a loan?",
    "Search the data for a mortgage-related issue involving Shellpoint Partners, LLC where the servicing remained with Newrez LLC. What was the core problem?",
    "If a consumer complains that Wells Fargo is incorrectly reporting a checking/savings account as a credit card on their credit profile, what is the typical company response or resolution outcome for such an issue?",
    "Based on the dataset, when consumers file complaints against credit reporting agencies like TransUnion for 'Improper use of your report', what legal sections or standard explanations do companies usually provide in their response?",
    "Summarize the primary reasons why consumers file complaints regarding 'Attempts to collect debt not owed' within this dataset sample. What is the most common underlying cause?",
    "What are the main arguments consumers use when accusing credit reporting companies (like Equifax or TransUnion) of violating their privacy rights under 15 U.S.C. 1681 Section 602?",
    "I went through my credit records and noticed fraudulent accounts that do not belong to me are still reposting on my file even after I disputed them. What insights or patterns do similar complaints in the database show regarding this situation?",
    "Are there any complaints in the dataset where credit bureaus or companies refuse to process disputes because they suspect it was filed by a third-party or a credit repair company rather than the consumer themselves?",
]

RETRIEVAL_GROUND_TRUTH: List[Dict[str, str]] = [
    {"company_contains": "U.S. BANCORP"},
    {"company_contains": "I.C. SYSTEM"},
    {"company_contains": "SHELLPOINT"},
    {"company_contains": "WELLS FARGO", "issue_contains": "credit card"},
    {"company_contains": "TRANSUNION", "issue_contains": "improper use"},
    {"issue_contains": "attempts to collect debt not owed"},
    {"company_contains": "EQUIFAX|TRANSUNION", "issue_contains": "privacy"},
    {"issue_contains": "fraud", "narrative_contains": "dispute"},
    {"issue_contains": "dispute", "narrative_contains": "third party|credit repair"},
]


def get_relevant_ids(df_source: pd.DataFrame, filt: Dict[str, str]) -> Set[str]:
    mask = pd.Series(True, index=df_source.index)
    if "company_contains" in filt:
        mask &= df_source["Company"].astype(str).str.contains(filt["company_contains"], case=False, na=False, regex=True)
    if "issue_contains" in filt:
        mask &= df_source["Issue"].astype(str).str.contains(filt["issue_contains"], case=False, na=False, regex=True)
    if "narrative_contains" in filt:
        mask &= df_source["narrative_clean"].astype(str).str.contains(filt["narrative_contains"], case=False, na=False, regex=True)
    return set(df_source.loc[mask, "Complaint ID"].astype(str))


def build_ground_truth_ids(df_processed: pd.DataFrame, filters: List[Dict[str, str]] = None) -> List[Set[str]]:
    filters = filters or RETRIEVAL_GROUND_TRUTH
    return [get_relevant_ids(df_processed, f) for f in filters]


def evaluate_retrieval(retriever, queries: List[str], ground_truth_ids: List[Set[str]], k: int = 5) -> Dict:
    recalls, reciprocal_ranks = [], []
    for query, rel_ids in zip(queries, ground_truth_ids):
        if not rel_ids:
            continue
        docs = retriever.vectorstore.similarity_search(query, k=k)
        retrieved_ids = [d.metadata.get("complaint_id") for d in docs[:k]]
        hits = [1 if rid in rel_ids else 0 for rid in retrieved_ids]
        recalls.append(sum(hits) / len(rel_ids))
        rr = 0.0
        for rank, is_hit in enumerate(hits, 1):
            if is_hit:
                rr = 1 / rank
                break
        reciprocal_ranks.append(rr)
    return {
        "k": k,
        "num_queries_evaluated": len(recalls),
        "Recall@K": round(sum(recalls) / len(recalls), 3) if recalls else 0.0,
        "MRR": round(sum(reciprocal_ranks) / len(reciprocal_ranks), 3) if reciprocal_ranks else 0.0,
    }


def run_retrieval_evaluation(retriever, df_processed: pd.DataFrame, k_values: List[int] = None) -> pd.DataFrame:
    k_values = k_values or [1, 3, 5]
    ground_truth_ids = build_ground_truth_ids(df_processed)
    rows = [evaluate_retrieval(retriever, TEST_QUERIES, ground_truth_ids, k=k) for k in k_values]
    return pd.DataFrame(rows)


### `app/evaluation/run_qualitative.py`
Runs the 9 fixed test queries through the *live* end-to-end RAG chain (retrieval + generation) and writes a human-readable `evaluation_report.txt` with each question and the bot's answer. Catches per-question exceptions so one failing query doesn't abort the whole run.

In [ ]:
%%writefile backend/app/evaluation/run_qualitative.py
"""Qualitative test-query run -- 9 fixed test queries through the live RAG chain."""
from pathlib import Path
from typing import List, Optional

from app.core.logging_config import configure_logging, get_logger
from app.evaluation.retrieval_metrics import TEST_QUERIES
from app.rag.chain import build_rag_chain

logger = get_logger(__name__)


def run_qualitative_eval(output_path: Optional[str] = None, queries: Optional[List[str]] = None) -> str:
    queries = queries or TEST_QUERIES
    output_path = output_path or "./evaluation_report.txt"
    chain = build_rag_chain()

    logger.info("Running RAG pipeline over %s test queries...", len(queries))
    eval_results = []
    for i, query in enumerate(queries, 1):
        logger.info("Answering question %s/%s...", i, len(queries))
        try:
            answer = chain.invoke(query)
        except Exception as e:  # noqa: BLE001
            answer = f"Error generating answer: {str(e)}"
        eval_results.append({"question_number": i, "question": query, "bot_answer": answer})

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n          RAG CHATBOT EVALUATION REPORT          \n" + "=" * 50 + "\n\n")
        for res in eval_results:
            f.write(f"### Q{res['question_number']}: {res['question']}\n")
            f.write(f"\U0001F916 Bot Answer:\n{res['bot_answer']}\n")
            f.write("-" * 50 + "\n\n")

    logger.info("Evaluation complete! Report saved to %s", output_path)
    return output_path


if __name__ == "__main__":
    configure_logging()
    run_qualitative_eval()


### `app/evaluation/run_all.py`
Standalone entry point (`python -m app.evaluation.run_all`) that runs all three evaluation steps independently of the API — qualitative test-query run, generation metrics (BLEU/ROUGE), and retrieval metrics (Recall@K/MRR) — printing and saving each as a report file (`evaluation_report.txt`, `metrics_report.csv`, `retrieval_evaluation_report.csv`).

In [ ]:
%%writefile backend/app/evaluation/run_all.py
"""
Standalone evaluation entry point -- runs qualitative + generation + retrieval
evaluation independently of the API, producing:
    evaluation_report.txt
    metrics_report.csv
    retrieval_evaluation_report.csv

Usage:
    python -m app.evaluation.run_all
"""
from app.core.logging_config import configure_logging, get_logger
from app.evaluation.generation_metrics import EVAL_DATASET, run_generation_evaluation
from app.evaluation.retrieval_metrics import run_retrieval_evaluation
from app.evaluation.run_qualitative import run_qualitative_eval
from app.utils.data_loader import load_dataframe_for_ground_truth
from app.vector_store.store import get_retriever

logger = get_logger(__name__)


def main() -> None:
    configure_logging()

    logger.info("== Step 1/3: Qualitative test-query run ==")
    run_qualitative_eval(output_path="./evaluation_report.txt")

    logger.info("== Step 2/3: Generation quality (BLEU / ROUGE) ==")
    retriever = get_retriever()
    df_metrics = run_generation_evaluation(retriever, EVAL_DATASET)
    print(df_metrics.to_markdown(index=False))
    df_metrics.to_csv("metrics_report.csv", index=False)

    logger.info("== Step 3/3: Retrieval quality (Recall@K, MRR) ==")
    df_processed = load_dataframe_for_ground_truth()
    retrieval_report_df = run_retrieval_evaluation(retriever, df_processed, k_values=[1, 3, 5])
    print(retrieval_report_df.to_markdown(index=False))
    retrieval_report_df.to_csv("retrieval_evaluation_report.csv", index=False)

    logger.info("All evaluation reports generated.")


if __name__ == "__main__":
    main()


## 11. Optimization experiments (prompt / embedding / chunking / retrieval strategy)

### `app/optimization/embedding_comparison.py`
Compares different Voyage AI embedding models (`voyage-3` vs `voyage-3-large`) on retrieval quality. `build_temp_retriever()` embeds a document sample into a disposable, uniquely-named Pinecone namespace (so experiments never touch the production 'default' namespace) and returns a retriever over it. `run_embedding_comparison()` runs Recall@K/MRR for each model on a 500-document sample and returns a results DataFrame.

In [ ]:
%%writefile backend/app/optimization/embedding_comparison.py
"""Embedding model comparison, backed by throwaway Pinecone namespaces and Voyage AI's hosted models."""
import time
from typing import List
from uuid import uuid4

import pandas as pd
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_voyageai import VoyageAIEmbeddings

from app.config.settings import get_settings
from app.core.exceptions import ConfigurationError
from app.evaluation.retrieval_metrics import evaluate_retrieval
from app.vector_store.builder import ensure_index_exists, get_pinecone_client

# The Pinecone index's dimension is fixed at creation time (EMBEDDING_DIMENSION=1024),
# so only Voyage models producing 1024-dim vectors can be compared against it here.
# voyage-3 is 1024-dim by default; voyage-3-large supports an explicit output_dimension.
EMBEDDING_MODELS = ["voyage-3", "voyage-3-large"]
_MODEL_KWARGS = {
    "voyage-3-large": {"output_dimension": 1024},
}


def build_temp_retriever(docs: List[Document], embedding_model: str, search_type: str = "similarity", k: int = 5):
    """
    Builds a retriever backed by a disposable Pinecone namespace (namespaced
    under the same index) so experiments don't touch the production
    'default' namespace.
    """
    settings = get_settings()
    if not settings.VOYAGE_API_KEY:
        raise ConfigurationError("VOYAGE_API_KEY is not set. Add it to your environment or .env file.")

    emb = VoyageAIEmbeddings(
        voyage_api_key=settings.VOYAGE_API_KEY,
        model=embedding_model,
        **_MODEL_KWARGS.get(embedding_model, {}),
    )

    pc = get_pinecone_client()
    ensure_index_exists(pc)
    index = pc.Index(settings.PINECONE_INDEX_NAME)

    namespace = f"exp-{uuid4().hex[:12]}"
    store = PineconeVectorStore(index=index, embedding=emb, namespace=namespace)
    ids = [d.metadata.get("complaint_id", f"tmp-{i}") for i, d in enumerate(docs)]
    store.add_documents(docs, ids=ids)
    time.sleep(1)  # brief settle time for Pinecone's eventual-consistency indexing

    return store.as_retriever(search_type=search_type, search_kwargs={"k": k})


def run_embedding_comparison(documents_sample: List[Document], test_queries, ground_truth_ids, embedding_models=None) -> pd.DataFrame:
    embedding_models = embedding_models or EMBEDDING_MODELS
    documents_sample = documents_sample[:500]
    embedding_results = []
    for model_name in embedding_models:
        temp_retriever = build_temp_retriever(docs=documents_sample, embedding_model=model_name, k=5)
        metrics = evaluate_retrieval(temp_retriever, test_queries, ground_truth_ids, k=5)
        metrics["Embedding Model"] = model_name
        embedding_results.append(metrics)
    return pd.DataFrame(embedding_results)


### `app/optimization/chunk_comparison.py`
Compares different text-chunking configurations (chunk size / overlap) for retrieval quality. Splits a document sample with `RecursiveCharacterTextSplitter` under each configuration, builds a temporary retriever for each, and reports Recall@K/MRR per configuration.

In [ ]:
%%writefile backend/app/optimization/chunk_comparison.py
"""Chunk size / overlap comparison, backed by throwaway Pinecone namespaces."""
from typing import List

import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from app.evaluation.retrieval_metrics import evaluate_retrieval
from app.optimization.embedding_comparison import build_temp_retriever

CHUNK_CONFIGS = [
    {"chunk_size": 300, "chunk_overlap": 30},
    {"chunk_size": 600, "chunk_overlap": 100},
]


def run_chunk_comparison(documents_sample: List[Document], test_queries, ground_truth_ids, chunk_configs=None) -> pd.DataFrame:
    chunk_configs = chunk_configs or CHUNK_CONFIGS
    chunk_results = []
    for cfg in chunk_configs:
        splitter = RecursiveCharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"])
        chunked_docs = splitter.split_documents(documents_sample)
        temp_retriever = build_temp_retriever(chunked_docs, "all-MiniLM-L6-v2", k=5)
        res = evaluate_retrieval(temp_retriever, test_queries, ground_truth_ids, k=5)
        res.update(cfg)
        chunk_results.append(res)
    return pd.DataFrame(chunk_results)


### `app/optimization/prompt_comparison.py`
Compares the two prompt versions (`v1` baseline vs `v2` role/constraint-based) from `app.rag.prompts.get_prompt()` on generation quality (ROUGE-1, ROUGE-L, BLEU) over the fixed evaluation dataset.

In [ ]:
%%writefile backend/app/optimization/prompt_comparison.py
"""Prompt engineering comparison (v1 baseline vs v2 role/constraint-based)."""
import pandas as pd

from app.evaluation.generation_metrics import calculate_metrics
from app.rag.prompts import get_prompt


def run_comparison(retriever, client, gemini_model: str, eval_dataset) -> pd.DataFrame:
    results = []
    for item in eval_dataset:
        docs = retriever.invoke(item["question"])
        context = "\n\n".join(doc.page_content for doc in docs)
        for version in ("v1", "v2"):
            full_prompt = get_prompt(context, item["question"], version)
            response = client.models.generate_content(model=gemini_model, contents=full_prompt)
            r1, rl, bleu = calculate_metrics(response.text, item["ground_truth"])
            results.append(
                {
                    "Version": version,
                    "Question": item["question"][:40] + "...",
                    "ROUGE-1": r1,
                    "ROUGE-L": rl,
                    "BLEU": bleu,
                }
            )
    return pd.DataFrame(results)


### `app/optimization/strategy_comparison.py`
Compares retrieval search strategies (`similarity` vs `mmr` — maximal marginal relevance) run directly against the production Pinecone index, reporting Recall@K/MRR for each strategy.

In [ ]:
%%writefile backend/app/optimization/strategy_comparison.py
"""Retrieval strategy (similarity vs MMR) comparison, run against the production Pinecone index."""
import pandas as pd

from app.evaluation.retrieval_metrics import evaluate_retrieval

STRATEGIES = ["similarity", "mmr"]


def run_strategy_comparison(vector_store, test_queries, ground_truth_ids, strategies=None) -> pd.DataFrame:
    strategies = strategies or STRATEGIES
    strategy_results = []
    for strategy in strategies:
        strat_retriever = vector_store.as_retriever(search_type=strategy, search_kwargs={"k": 5})
        res = evaluate_retrieval(strat_retriever, test_queries, ground_truth_ids, k=5)
        res["strategy"] = strategy
        strategy_results.append(res)
    return pd.DataFrame(strategy_results)


### `app/optimization/run_optimization.py`
Standalone entry point (`python -m app.optimization.run_optimization`) that runs all four optimization experiments in sequence — retriever k comparison, prompt engineering comparison, embedding model comparison, chunk size/overlap comparison, and retrieval strategy comparison — and writes a combined Markdown report, `optimization_experiments_report.md`.

In [ ]:
%%writefile backend/app/optimization/run_optimization.py
"""
Standalone optimization entry point -- runs all four experiments (prompt,
embedding model, chunk size/overlap, retrieval strategy) and writes
`optimization_experiments_report.md`. Independent of the API.

Usage:
    python -m app.optimization.run_optimization
"""
from app.core.logging_config import configure_logging, get_logger
from app.evaluation.generation_metrics import EVAL_DATASET
from app.evaluation.retrieval_metrics import TEST_QUERIES, build_ground_truth_ids, run_retrieval_evaluation
from app.optimization.chunk_comparison import run_chunk_comparison
from app.optimization.embedding_comparison import run_embedding_comparison
from app.optimization.prompt_comparison import run_comparison
from app.optimization.strategy_comparison import run_strategy_comparison
from app.utils.data_loader import load_dataframe_for_ground_truth, load_documents
from app.vector_store.store import get_retriever, get_vector_store

logger = get_logger(__name__)


def main() -> None:
    configure_logging()

    df_processed = load_dataframe_for_ground_truth()
    ground_truth_ids = build_ground_truth_ids(df_processed)
    documents_sample = load_documents(max_rows=500)

    retriever = get_retriever()
    vector_store = get_vector_store()

    logger.info("Retriever k comparison (k=1,3,5)...")
    retrieval_report_df = run_retrieval_evaluation(retriever, df_processed, k_values=[1, 3, 5])

    logger.info("Prompt engineering comparison (v1 vs v2)...")
    comparison_df = run_comparison(retriever, EVAL_DATASET)
    comparison_df.to_csv("prompt_comparison_report.csv", index=False)

    logger.info("Embedding model comparison (MiniLM vs mpnet)...")
    embedding_results_df = run_embedding_comparison(documents_sample, TEST_QUERIES, ground_truth_ids)

    logger.info("Chunk size / overlap comparison...")
    chunk_results_df = run_chunk_comparison(documents_sample, TEST_QUERIES, ground_truth_ids)

    logger.info("Retrieval strategy comparison (similarity vs MMR)...")
    strategy_results_df = run_strategy_comparison(vector_store, TEST_QUERIES, ground_truth_ids)

    with open("optimization_experiments_report.md", "w", encoding="utf-8") as f:
        f.write("# Optimization Experiments\n\n")
        f.write("## Retriever k Comparison\n\n" + retrieval_report_df.to_markdown(index=False) + "\n\n")
        f.write("## Prompt Engineering Comparison\n\n" + comparison_df.to_markdown(index=False) + "\n\n")
        f.write("## Embedding Model Comparison\n\n" + embedding_results_df.to_markdown(index=False) + "\n\n")
        f.write("## Chunk Size / Overlap Comparison\n\n" + chunk_results_df.to_markdown(index=False) + "\n\n")
        f.write("## Retrieval Strategy Comparison\n\n" + strategy_results_df.to_markdown(index=False) + "\n")

    logger.info("All optimization experiments complete. Report saved to optimization_experiments_report.md")


if __name__ == "__main__":
    main()


## Notes

- Every file above was copied **verbatim** from `backend/app/...` — nothing was shortened, summarized, or omitted.
- The `%%writefile` magic at the top of each code cell means running this notebook top-to-bottom will physically recreate the `backend/app` folder tree in the notebook's working directory.
- The dataset file `backend/data/processed_corpus_5000.csv` (~13 MB) is data, not code, so it is not duplicated here — copy it separately if you need to run the pipeline end-to-end.
